# Face Detection Using Hear Cascade Classifications 

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
os.chdir(r"C:\Users\DT1\Desktop\proj-python\models")

In [2]:
model=cv2.CascadeClassifier("haarcascade_frontalface_default.xml")

if model.empty():
    raise IOError("Could not load haarcascade_frontalface_default.xml")

cap=cv2.VideoCapture(0)

cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
cap.set(cv2.CAP_PROP_FPS, 30)
while cap.isOpened():
    ret,frame=cap.read()
    if ret:
        gray=cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
        blur=cv2.GaussianBlur(gray,(5,5),0)
        blur=cv2.medianBlur(blur,5)
    

        kernal=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5))
        mask=cv2.morphologyEx(blur,cv2.MORPH_OPEN,kernal,3)
        mask=cv2.morphologyEx(mask,cv2.MORPH_CLOSE,kernal,3)

        face=model.detectMultiScale(mask,1.1,5)
        for (x,y,w,h) in face:
            cv2.rectangle(frame,(x,y),(x+w,y+h),(0,255,0),2,cv2.LINE_AA)
            cv2.putText(frame,"FACE",(x, y - 10),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0, 255, 0),2,cv2.LINE_AA)
    
        cv2.imshow("Detection Face",frame)
        if cv2.waitKey(1) & 0xFF ==ord('q'):
            break
cap.release()
cv2.destroyAllWindows()

# Simple difference between Cany and Cascade detection

In [3]:
import cv2

# =========================
# Haar Cascade
# =========================
model = cv2.CascadeClassifier(
    "haarcascade_frontalface_default.xml"
)

if model.empty():
    raise IOError(
        "Could not load haarcascade_frontalface_default.xml"
    )


# =========================
# Camera
# =========================
cap = cv2.VideoCapture(0)

cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
cap.set(cv2.CAP_PROP_FPS, 30)


# =========================
# Kernel
# =========================
kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE,
    (5, 5)
)


# =========================
# Main loop
# =========================
while cap.isOpened():

    ret, frame = cap.read()

    if not ret:
        break

    # Mirror
    frame = cv2.flip(frame, 1)

    # -------------------------
    # Grayscale
    # -------------------------
    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY
    )

    # -------------------------
    # Gaussian Blur
    # -------------------------
    blur = cv2.GaussianBlur(
        gray,
        (5, 5),
        0
    )

    # =========================
    # FACE DETECTION
    # =========================
    faces = model.detectMultiScale(
        blur,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(60, 60)
    )

    # Draw faces
    for (x, y, w, h) in faces:

        cv2.rectangle(
            frame,
            (x, y),
            (x + w, y + h),
            (0, 255, 0),
            2,
            cv2.LINE_AA
        )

        cv2.putText(
            frame,
            "FACE",
            (x, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2,
            cv2.LINE_AA
        )

    # =========================
    # CANNY BRANCH
    # =========================

    edges = cv2.Canny(
        blur,
        50,
        155
    )

    # Remove small noise
    edges = cv2.morphologyEx(
        edges,
        cv2.MORPH_OPEN,
        kernel,
        iterations=1
    )

    # Fill small gaps
    edges = cv2.morphologyEx(
        edges,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=1
    )

    # =========================
    # Display
    # =========================

    cv2.imshow(
        "Face Detection",
        frame
    )

    cv2.imshow(
        "Canny + Morphology",
        edges
    )

    # Quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break


# =========================
# Cleanup
# =========================
cap.release()
cv2.destroyAllWindows()

In [4]:
import cv2
import numpy as np
import time
from datetime import datetime


# =========================================================
# 1. LOAD FACE DETECTOR
# =========================================================

face_model = cv2.CascadeClassifier(
    "haarcascade_frontalface_default.xml"
)

if face_model.empty():
    raise IOError(
        "Could not load haarcascade_frontalface_default.xml"
    )


# =========================================================
# 2. CAMERA
# =========================================================

cap = cv2.VideoCapture(0)

cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
cap.set(cv2.CAP_PROP_FPS, 30)

if not cap.isOpened():
    raise IOError("Could not open camera")


# =========================================================
# 3. PARAMETERS
# =========================================================

SCALE_FACTOR = 1.08
MIN_NEIGHBORS = 6
MIN_SIZE = (70, 70)

CANNY_LOW = 50
CANNY_HIGH = 150

kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE,
    (3, 3)
)


# =========================================================
# 4. CLAHE
# =========================================================

clahe = cv2.createCLAHE(
    clipLimit=2.0,
    tileGridSize=(8, 8)
)


# =========================================================
# 5. VARIABLES
# =========================================================

start_time = time.time()

total_faces_seen = 0
max_faces = 0

previous_center = None
motion_pixels = 0

last_screenshot = 0


# =========================================================
# 6. MAIN LOOP
# =========================================================

while True:

    ret, frame = cap.read()

    if not ret:
        print("Frame error")
        break

    # Mirror camera
    frame = cv2.flip(frame, 1)

    # -----------------------------------------------------
    # IMAGE PREPROCESSING
    # -----------------------------------------------------

    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY
    )

    blur = cv2.GaussianBlur(
        gray,
        (5, 5),
        0
    )

    enhanced = clahe.apply(blur)


    # =====================================================
    # FACE DETECTION
    # =====================================================

    faces = face_model.detectMultiScale(
        enhanced,
        scaleFactor=SCALE_FACTOR,
        minNeighbors=MIN_NEIGHBORS,
        minSize=MIN_SIZE
    )


    # =====================================================
    # STATISTICS
    # =====================================================

    face_count = len(faces)

    if face_count > max_faces:
        max_faces = face_count

    if face_count > 0:
        total_faces_seen += 1


    # =====================================================
    # PROCESS EACH FACE
    # =====================================================

    for index, (x, y, w, h) in enumerate(faces):

        # -------------------------------------------------
        # FACE ROI
        # -------------------------------------------------

        face_roi = gray[
            y:y+h,
            x:x+w
        ]

        if face_roi.size == 0:
            continue


        # =================================================
        # CANNY
        # =================================================

        face_blur = cv2.GaussianBlur(
            face_roi,
            (5, 5),
            0
        )

        edges = cv2.Canny(
            face_blur,
            CANNY_LOW,
            CANNY_HIGH
        )


        # =================================================
        # MORPHOLOGY
        # =================================================

        edges = cv2.morphologyEx(
            edges,
            cv2.MORPH_OPEN,
            kernel,
            iterations=1
        )

        edges = cv2.morphologyEx(
            edges,
            cv2.MORPH_CLOSE,
            kernel,
            iterations=1
        )


        # =================================================
        # EDGE DENSITY
        # =================================================

        edge_pixels = np.count_nonzero(edges)

        total_pixels = edges.shape[0] * edges.shape[1]

        edge_density = (
            edge_pixels / total_pixels
        ) * 100


        # =================================================
        # IMAGE QUALITY
        # =================================================

        sharpness = cv2.Laplacian(
            face_roi,
            cv2.CV_64F
        ).var()


        # =================================================
        # QUALITY CLASSIFICATION
        # =================================================

        if sharpness > 250:
            quality = "GOOD"

        elif sharpness > 100:
            quality = "MEDIUM"

        else:
            quality = "BLUR"


        # =================================================
        # CENTER
        # =================================================

        cx = x + w // 2
        cy = y + h // 2


        # =================================================
        # MOTION
        # =================================================

        motion = "STABLE"

        if previous_center is not None:

            dx = cx - previous_center[0]
            dy = cy - previous_center[1]

            distance = np.sqrt(
                dx**2 + dy**2
            )

            if distance > 15:
                motion = "MOVING"

        previous_center = (cx, cy)


        # =================================================
        # DRAW FACE
        # =================================================

        cv2.rectangle(
            frame,
            (x, y),
            (x+w, y+h),
            (0, 255, 0),
            2,
            cv2.LINE_AA
        )


        # =================================================
        # CENTER
        # =================================================

        cv2.circle(
            frame,
            (cx, cy),
            5,
            (0, 0, 255),
            -1
        )


        # =================================================
        # INFORMATION
        # =================================================

        cv2.putText(
            frame,
            f"FACE {index + 1}",
            (x, y - 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (0, 255, 0),
            2,
            cv2.LINE_AA
        )

        cv2.putText(
            frame,
            f"Quality: {quality}",
            (x, y + h + 25),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (255, 255, 255),
            2,
            cv2.LINE_AA
        )

        cv2.putText(
            frame,
            f"Sharpness: {int(sharpness)}",
            (x, y + h + 48),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (255, 255, 255),
            2,
            cv2.LINE_AA
        )

        cv2.putText(
            frame,
            f"Edges: {edge_density:.1f}%",
            (x, y + h + 71),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (255, 255, 255),
            2,
            cv2.LINE_AA
        )

        cv2.putText(
            frame,
            f"Motion: {motion}",
            (x, y + h + 94),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (255, 255, 255),
            2,
            cv2.LINE_AA
        )


        # =================================================
        # SHOW FACE EDGE ROI
        # =================================================

        edge_display = cv2.resize(
            edges,
            (250, 200)
        )

        cv2.imshow(
            "Face Edges",
            edge_display
        )


    # =====================================================
    # DASHBOARD
    # =====================================================

    elapsed = time.time() - start_time

    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps <= 0:
        fps = 30


    # Dashboard background
    cv2.rectangle(
        frame,
        (10, 10),
        (300, 145),
        (20, 20, 20),
        -1
    )


    cv2.putText(
        frame,
        f"Faces: {face_count}",
        (25, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (0, 255, 0),
        2
    )

    cv2.putText(
        frame,
        f"Max Faces: {max_faces}",
        (25, 68),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        f"FPS: {fps:.1f}",
        (25, 96),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        f"Time: {elapsed:.0f}s",
        (25, 124),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2
    )


    # =====================================================
    # CURRENT DATE/TIME
    # =====================================================

    now = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    cv2.putText(
        frame,
        now,
        (950, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 255, 255),
        2
    )


    # =====================================================
    # DISPLAY
    # =====================================================

    cv2.imshow(
        "SMART FACE ANALYZER",
        frame
    )


    # =====================================================
    # KEYBOARD
    # =====================================================

    key = cv2.waitKey(1) & 0xFF


    # Q → Quit
    if key == ord("q"):
        break


    # S → Save screenshot
    if key == ord("s"):

        filename = (
            "face_capture_"
            + datetime.now().strftime(
                "%Y%m%d_%H%M%S"
            )
            + ".jpg"
        )

        cv2.imwrite(
            filename,
            frame
        )

        print(
            f"Saved: {filename}"
        )


# =========================================================
# CLEANUP
# =========================================================

cap.release()
cv2.destroyAllWindows()